# NOAA POES (NOAA-15, -16, -17, -18, -19) / AVHRR/3 (Advanced Very High Resolution Radiometer)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=noaa/noaa-poes-avhrr). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B: calibrated, geolocated counts, radiances and brightness temperatures per orbit in LAC (1.1 km), GAC (4 km) or HRPT formats.

Reads a local AVHRR L1B (GAC/LAC) file ordered from NOAA CLASS with satpy and plots channel 4. Without a file in ./data it only prints instructions, because L1B data must be ordered first.

In [ ]:
%pip install -q satpy pygac matplotlib

In [ ]:
from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

# AVHRR L1B files ordered from NOAA CLASS (or received by direct broadcast) go in ./data
files = sorted(glob("data/**/*", recursive=True))
files = [f for f in files if "NSS.GHRR" in f or "NSS.LHRR" in f or f.endswith(".l1b")]
if not files:
    print("No AVHRR L1B file found in ./data: order one from NOAA CLASS (see the Original Method tab) and re-run")
else:
    scn = Scene(filenames=files[:1], reader="avhrr_l1b_gaclac")
    scn.load(["4"])
    scn["4"].plot(cmap="gray_r")
    plt.title("AVHRR channel 4 (10.8 um)")
    plt.show()

## L3

Level 3: NOAA Climate Data Records from AVHRR: daily 0.05 degree surface reflectance and NDVI (AVHRR Land CDR, 1981-present) and daily SST (OISST).

Reads one daily NDVI file of the NOAA AVHRR Land CDR (global, 0.05 degrees) from the public NOAA bucket on AWS (no credentials) and plots it.

In [ ]:
%pip install -q s3fs xarray h5netcdf matplotlib

In [ ]:
import matplotlib.pyplot as plt
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=True)                # public NOAA bucket, no credentials needed
BUCKET, YEAR, DAY = "noaa-cdr-ndvi-pds", 2020, "20200601"
files = [f for f in fs.ls("%s/data/%d/" % (BUCKET, YEAR)) if DAY in f]
print(files)
if not files:
    raise SystemExit("No file for this day: list the prefix to see what is available")

with fs.open(files[0]) as fobj:
    ds = xr.open_dataset(fobj, engine="h5netcdf").load()
print(ds)
ds["NDVI"].isel(time=0)[::5, ::5].plot(cmap="YlGn", vmin=0, vmax=1)
plt.title("NOAA CDR AVHRR NDVI, 1 June 2020 (0.05 degrees)")
plt.show()